Step 1: Data Clearning & Preparation

In [1]:
import pandas as pd
import numpy as np

# Đọc file Superstore (thường là .csv hoặc .xls, để ý encoding)
df = pd.read_csv("Sample_Superstore.csv", encoding="latin1")
# Nếu file Excel: df = pd.read_excel("Superstore.xlsx")

print(df.shape)
df.head()
df.info()

(9994, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 1

In [2]:
# parse ngay thang dung dinh dang
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

In [3]:
# kiểm tra dữ liệu thiếu & trùng lặp
print(df.isnull().sum()) #kiem tra null

# kiem tra trung lap toan bo dong
print(df.duplicated().sum())

#Postal code thuong bi thieu (zip khong du 5 so) kiem tra de xua ly
df[df['Postal Code'].isnull()]

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
0


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [4]:
# tính thời gian giao hàng
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

#bieen loi nhuan
df['Profit Margin'] = df['Profit'] / df['Sales']

# chi nho nam va thang
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month


In [5]:
# join toa do Zip code
zip_geo = pd.read_csv("US_zipCode/uszips.csv")
zip_geo['zip'] = zip_geo['zip'].astype(str).str.zfill(5)
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)
df = df.merge(zip_geo[['zip', 'lat', 'lng']], left_on = 'Postal Code', right_on = 'zip', how = 'left')

print(df['lat'].isnull().sum())

0


Step 2: Customer & Regional Analysis

In [6]:
state_summary = df.groupby('State').agg(
    Total_Sales = ('Sales', 'sum'),
    Total_profit = ('Profit', 'sum'),
    Order_count =('Order ID', 'nunique'),
    Avg_Profit_Margin = ('Profit Margin', 'mean')
).sort_values('Total_Sales', ascending=False)

print(state_summary.head(10))
print(state_summary.tail(10))

              Total_Sales  Total_profit  Order_count  Avg_Profit_Margin
State                                                                  
California    457687.6315    76381.3871         1021           0.278334
New York      310876.2710    74038.5486          562           0.298366
Texas         170188.0458   -25729.3563          487          -0.342011
Washington    138641.2700    33402.6517          256           0.276354
Pennsylvania  116511.9140   -15559.9603          288          -0.086013
Florida        89473.7080    -3399.3017          200          -0.017953
Illinois       80166.1010   -12607.8870          276          -0.391677
Ohio           78258.1360   -16971.3766          236          -0.073808
Michigan       76269.6140    24463.1876          117           0.333390
Virginia       70636.7200    18597.9504          115           0.332009
                      Total_Sales  Total_profit  Order_count  \
State                                                          
New Mexi

In [7]:
df.groupby(['State', 'Sub-Category']).agg(
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean')
).sort_values('Total_Profit').head(15)

,,Total_Profit,Avg_Discount
State,Sub-Category,,
Texas,Binders,-14705.0738,0.8
Ohio,Machines,-11770.9447,0.7
Illinois,Binders,-7204.3242,0.8
Texas,Appliances,-6147.2225,0.8
North Carolina,Machines,-5384.8086,0.5
Pennsylvania,Binders,-4570.9750,0.7
New York,Tables,-4535.6408,0.4
Colorado,Machines,-4384.2554,0.7
Illinois,Tables,-4309.7447,0.5


In [ ]:
losing_states = state_summary[state_summary['Total_profit'] < 0]
print(losing_states)
# nhung bang lo 

                Total_Sales  Total_profit  Order_count  Avg_Profit_Margin
State                                                                    
Texas           170188.0458   -25729.3563          487          -0.342011
Pennsylvania    116511.9140   -15559.9603          288          -0.086013
Florida          89473.7080    -3399.3017          200          -0.017953
Illinois         80166.1010   -12607.8870          276          -0.391677
Ohio             78258.1360   -16971.3766          236          -0.073808
North Carolina   55603.1640    -7490.9122          136           0.007751
Arizona          35282.0010    -3427.9246          108          -0.066399
Colorado         32108.1180    -6527.8579           79          -0.123755
Tennessee        30661.8730    -5341.6936           91          -0.016794
Oregon           17431.1500    -1190.4705           56          -0.050094


In [9]:
# AOV theo region
aov_region = df.groupby('Region').agg(
    Total_Sales = ('Sales','sum'),
    Order_Count = ('Order ID', 'nunique')
)
aov_region['AOV'] = aov_region['Total_Sales'] / aov_region['Order_Count']
print(aov_region)

#AOV theo Segment
aov_segment = df.groupby('Segment').agg(
    Total_sales = ('Sales', 'sum'),
    Order_Count = ('Order ID', 'nunique')
)
aov_segment['AOV'] = aov_segment['Total_sales'] / aov_segment['Order_Count']
print(aov_segment)

         Total_Sales  Order_Count         AOV
Region                                       
Central  501239.8908         1175  426.587141
East     678781.2400         1401  484.497673
South    391721.9050          822  476.547330
West     725457.8245         1611  450.315223
              Total_sales  Order_Count         AOV
Segment                                           
Consumer     1.161401e+06         2586  449.111116
Corporate    7.061464e+05         1514  466.411075
Home Office  4.296531e+05          909  472.665730


In [10]:
import datetime as dt

df['Order Date'] = pd.to_datetime(df['Order Date'])
snapshot_date =  df['Order Date'].max() + dt.timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    recency = ('Order Date', lambda x: (snapshot_date - x.max()).days),
    frequency = ('Order ID', 'nunique'),
    monetary = ('Sales', 'sum')
)
print(rfm.describe())

# phan nhom
rfm['R_Score'] = pd.qcut(rfm['recency'], 4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['monetary'], 4, labels=[1,2,3,4])
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

def segment_customer(row):
    if row['R_Score'] >= 3 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
        return 'VIP'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'At Risk / Churned'
    elif row['F_Score'] >= 3:
        return 'Loyal'
    else:
        return 'Regular'
rfm['segment'] = rfm.apply(segment_customer, axis=1)
print(rfm['segment'].value_counts())


           recency   frequency      monetary
count   793.000000  793.000000    793.000000
mean    147.802018    6.316520   2896.848500
std     186.211051    2.550885   2628.670117
min       1.000000    1.000000      4.833000
25%      31.000000    5.000000   1146.050000
50%      76.000000    6.000000   2256.394000
75%     184.000000    8.000000   3785.276000
max    1166.000000   17.000000  25043.050000
segment
At Risk / Churned    238
Loyal                220
VIP                  176
Regular              159
Name: count, dtype: int64


In [ ]:
customer_state = df.groupby('Customer ID')['State'].agg(lambda x: x.mode()[0])
rfm = rfm.join(customer_state)

segment_by_state = pd.crosstab(rfm['State'], rfm['segment'])
print(segment_by_state)

segment               At Risk / Churned  Loyal  Regular  VIP
State                                                       
Alabama                               2      0        0    0
Arizona                               5      6        4    5
Arkansas                              1      1        3    2
California                           62     71       49   62
Colorado                              6      5        2    4
Connecticut                           3      0        1    0
Delaware                              3      3        2    0
District of Columbia                  0      1        0    0
Florida                               8      6        9    7
Georgia                               3      5        3    1
Illinois                             11     10        5    7
Indiana                               2      4        3    3
Iowa                                  1      0        1    0
Kentucky                              3      2        5    3
Louisiana               

In [17]:
df.to_csv('superstore_cleaned2.csv', index=False)
rfm.to_csv("customer_rfm_segments.csv")